<b> Estadística | El horizonte cambia la distribución. </b>  Simula 20_000 retornos diarios con una Student-\(t\), df=4, y escala a aproximadamente 1% de volatilidad diaria. Construye retornos acumulados no solapados a horizontes 
$ (H=\{1,5,20\})$:

$$ R_{t,H}=\prod_{j=0}^{H-1}(1+r_{t+j})-1. $$

Para cada horizonte calcula solamente std, skew, kurtosis y P(R_H < -2*std_H). 
No anualices. 

La pregunta es: ¿la distribución a 20 días es simplemente “la distribución de un día multiplicada por 20”? Explica qué cambió al agregar retornos y qué propiedades todavía podrían impedir una aproximación normal razonable en datos financieros reales.

In [9]:
import numpy as np 
import scipy.stats as stats
import pandas as pd 
# generamos serie de retornos usando una distribucion t-student con 4 grados de libertad 
# generador reproducible 
rng = np.random.default_rng(seed = 42)
retornos_diarios = rng.standard_t(df = 4, size = 20000)
# volatilidad diaria : 1% , escalar la serie  
vol_1d = 0.01 
retornos_diarios = (retornos_diarios / retornos_diarios.std(ddof = 1)) * vol_1d
horizontes = [1,5,20]
retornos_acumulados = {}
for h in horizontes:
    bloques = retornos_diarios.reshape(-1,h)
    # retornos acumulados no zolapados para 1dia,5, 20 
    retornos_h = np.prod(1+bloques, axis = 1) - 1
    retornos_acumulados[f'{h}d'] =retornos_h

def metricas(retornos):
    std_h = retornos.std(ddof=1)
    return {'std' : std_h, 
            'skew': stats.skew(retornos),
            'kurtosis': stats.kurtosis(retornos), 
            'P(R_H < -2*std_H)': (retornos < -2 * std_h).mean()}
resultados = []
for horizonte, retornos in retornos_acumulados.items(): 
    fila = {'horizontes': horizonte}
    fila.update(metricas(retornos))
    resultados.append(fila)
df_metricas = pd.DataFrame(resultados)
df_metricas

,horizontes,std,skew,kurtosis,P(R_H < -2*std_H)
0,1d,0.010000,0.953190,25.518796,0.0234
1,5d,0.022188,0.518217,5.347729,0.0215
2,20d,0.044496,0.127074,0.985706,0.0240


1. la distribución a 20 días es simplemente “la distribución de un día multiplicada por 20”? 

   R: No dado a diferentes puntos; el retorno a 20 dias es el producto compuesto de 20 retornos diarios consecutivos, no el retorno diario multiplicado por 20. segunda observacióm, la desviación estandar partiendo del suspuesto que los retornos son independientes, la desviacion o volatilidad no crece de manera lineal, por ende no puede simplemente ser la misma distribucion pero multiplicada por 20. 

2. Explica qué cambió al agregar retornos y qué propiedades todavía podrían impedir una aproximación normal razonable en datos financieros reales.

   R: Al agregar retornos diarios en bloques no solapados, cada observación pasa a representar un período más largo. Por ello disminuye el número de observaciones disponibles, de 20,000 a 1,000 para el horizonte de 20 días, y cambian la desviación estándar, asimetría y curtosis. El TLC sugiere que, si los retornos fueran independientes y tuvieran varianza finita, la distribución agregada tendería a aproximarse a una normal conforme aumenta el horizonte. Sin embargo, en datos financieros reales esa aproximación puede seguir siendo pobre por colas pesadas, asimetría, autocorrelación, volatilidad cambiante (volatility clustering), saltos de precio y cambios de régimen

<b> Portfolio/Risk Engine | El mismo escenario, tres horizontes. </b> Define tres activos y una representación pequeña:

$$ R\in\mathbb{R}^{4\times3\times3}, $$

donde los ejes son scenario × asset × horizon y los horizontes son 1D, 5D, 20D. Inventa cuatro escenarios coherentes —normal, equity selloff, rates shock, risk-on— y pesos \(w=(0.4,0.35,0.25)\). Calcula

$$ R_p^{(s,h)}=\sum_i w_iR_{s,i,h} $$

y genera una tabla scenario × horizon con el retorno del portfolio. Después identifica qué escenario es el peor en cada horizonte. Pregunta final: 
 - ¿por qué tu Risk Engine debería recibir explícitamente el horizonte \(H\), en vez de almacenar simplemente “el riesgo del portfolio”?